# MOT-MIC Example

This notebook demonstrates the full MOT-MIC workflow on a small synthetic dataset that mimics paired primary tumor cells and multiple metastatic sites.

For real analyses, replace the simulated matrices with tumor-cell matrices from GSE173958, GSE249057, GSE178318, or GSE277783 after QC and malignant-cell filtering.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(repo_root))

from motmic import MOTMIC, evaluate_binary_labels, simulate_metastasis_dataset
from motmic.interpret import rank_genes_with_shap

primary, metastases, truth = simulate_metastasis_dataset(random_state=7)
primary.shape, {k: v.shape for k, v in metastases.items()}, truth.head()

## Fit MOT-MIC

The estimator learns a shared latent space, maps primary cells to each metastatic site with unbalanced optimal transport, applies top-k origin filtering, and returns organ-specific MIC scores.

In [ ]:
model = MOTMIC(n_components=15, epsilon=0.08, rho=1.2, top_k=1)
result = model.fit_predict(primary, metastases)
scores = result.to_frame()
scores.head()

## Validate Against Lineage-Like Labels

In real GSE173958 analysis, replace `truth['true_MIC']` with aggressive clone labels or clone dissemination labels.

In [ ]:
metrics = evaluate_binary_labels(result.pan_score, truth['true_MIC'])
metrics

## Rank Candidate Genes

SHAP is used when available. If SHAP is unavailable, the function falls back to permutation importance so the workflow still runs.

In [ ]:
high_mic = result.pan_score >= result.pan_score.quantile(0.8)
ranking = rank_genes_with_shap(primary, high_mic.astype(int))
ranking.head(20)

## Save Results

The same tables are produced by `scripts/run_example.py`.

In [ ]:
from pathlib import Path
Path('../results').mkdir(exist_ok=True)
scores.to_csv('../results/notebook_motmic_scores.csv')
ranking.to_csv('../results/notebook_gene_ranking.csv', index=False)